In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
NVIDIA GeForce RTX 5070 Laptop GPU


In [2]:
import shutil
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# =========================================================
# 1. PATHS — CHANGE THESE
# =========================================================
rdd_root = Path(r"D:\Big_Data\2020YOLO")
sovit_root = Path(r"D:\Big_Data\Dataset 1 (Simplex)\Dataset 1 (Simplex)")

class_review = rdd_root / "class_review"
output_root = rdd_root / "final_cnn_dataset"

# Sovit folders
sovit_positive_dir = sovit_root / "Train data" / "Positive data"
sovit_negative_dir = sovit_root / "Train data" / "Negative data"   

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# =========================================================
# 2. RDD SOURCE FOLDERS
# =========================================================
rdd_pothole_raw = class_review / "pothole" / "raw_images"
rdd_alligator_raw = class_review / "alligator" / "raw_images"
rdd_long_raw = class_review / "longitudinal" / "raw_images"
rdd_trans_raw = class_review / "transverse" / "raw_images"

# =========================================================
# 3. HELPERS
# =========================================================
def list_images(folder: Path):
    if not folder.exists():
        return []
    return [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in image_exts]

def safe_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

def count_images(folder: Path):
    if not folder.exists():
        return 0
    return len([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in image_exts])

# =========================================================
# 4. BUILD RDD POOLS
# =========================================================
rdd_pothole_files = list_images(rdd_pothole_raw)
rdd_alligator_files = list_images(rdd_alligator_raw)
rdd_long_files = list_images(rdd_long_raw)
rdd_trans_files = list_images(rdd_trans_raw)

pothole_names = {p.name for p in rdd_pothole_files}
alligator_names = {p.name for p in rdd_alligator_files}

# clean negative:
# from longitudinal + transverse
# remove anything also appearing in pothole or alligator
negative_dict = {}

for p in rdd_long_files + rdd_trans_files:
    if p.name in pothole_names or p.name in alligator_names:
        continue
    if p.name not in negative_dict:
        negative_dict[p.name] = p

rdd_negative_files = list(negative_dict.values())

print("RDD pothole:", len(rdd_pothole_files))
print("RDD clean negative:", len(rdd_negative_files))
print("RDD alligator challenge:", len(rdd_alligator_files))

# =========================================================
# 5. SPLIT RDD ONLY
#    train/val/test 都先来自 RDD
# =========================================================
rdd_samples = []

for p in rdd_pothole_files:
    rdd_samples.append({
        "img_path": str(p),
        "label": 1,
        "source": "rdd"
    })

for p in rdd_negative_files:
    rdd_samples.append({
        "img_path": str(p),
        "label": 0,
        "source": "rdd"
    })

rdd_df = pd.DataFrame(rdd_samples)

train_df, temp_df = train_test_split(
    rdd_df,
    test_size=0.20,
    random_state=42,
    stratify=rdd_df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

print("\nRDD split sizes")
print("train:", len(train_df))
print("val  :", len(val_df))
print("test :", len(test_df))

# =========================================================
# 6. SOVIT POSITIVE ONLY
# =========================================================
sovit_positive_files = list_images(sovit_positive_dir)
print("\nSovit positive images found:", len(sovit_positive_files))

# =========================================================
# 7. CREATE OUTPUT FOLDERS
# =========================================================
for split in ["train", "val", "test"]:
    (output_root / split / "pothole").mkdir(parents=True, exist_ok=True)
    (output_root / split / "non_pothole").mkdir(parents=True, exist_ok=True)

(output_root / "challenge_alligator" / "alligator").mkdir(parents=True, exist_ok=True)

# =========================================================
# 8. EXPORT RDD SPLITS
# =========================================================
def export_df(df: pd.DataFrame):
    counts = {"pothole": 0, "non_pothole": 0}

    for _, row in df.iterrows():
        src = Path(row["img_path"])
        split = row["split"]
        cls = "pothole" if row["label"] == 1 else "non_pothole"

        new_name = f"rdd_{src.name}"
        dst = output_root / split / cls / new_name
        safe_copy(src, dst)

        counts[cls] += 1

    return counts

train_counts = export_df(train_df)
val_counts = export_df(val_df)
test_counts = export_df(test_df)

print("\nRDD exported")
print("train:", train_counts)
print("val  :", val_counts)
print("test :", test_counts)

# =========================================================
# 9. ADD SOVIT POSITIVE ONLY TO TRAIN/POTHOLE
# =========================================================
sovit_added = 0
for src in sovit_positive_files:
    new_name = f"sovit_{src.name}"
    dst = output_root / "train" / "pothole" / new_name
    safe_copy(src, dst)
    sovit_added += 1

print("\nSovit positives added to train/pothole:", sovit_added)

# =========================================================
# 10. EXPORT ALLIGATOR CHALLENGE SET
# =========================================================
alligator_added = 0
for src in rdd_alligator_files:
    new_name = f"rdd_alligator_{src.name}"
    dst = output_root / "challenge_alligator" / "alligator" / new_name
    safe_copy(src, dst)
    alligator_added += 1

print("Alligator challenge images:", alligator_added)

# =========================================================
# 11. SAVE MANIFEST
# =========================================================
manifest_rows = []

for _, row in train_df.iterrows():
    manifest_rows.append({
        "img_path": row["img_path"],
        "split": "train",
        "class_name": "pothole" if row["label"] == 1 else "non_pothole",
        "source": "rdd"
    })

for _, row in val_df.iterrows():
    manifest_rows.append({
        "img_path": row["img_path"],
        "split": "val",
        "class_name": "pothole" if row["label"] == 1 else "non_pothole",
        "source": "rdd"
    })

for _, row in test_df.iterrows():
    manifest_rows.append({
        "img_path": row["img_path"],
        "split": "test",
        "class_name": "pothole" if row["label"] == 1 else "non_pothole",
        "source": "rdd"
    })

for src in sovit_positive_files:
    manifest_rows.append({
        "img_path": str(src),
        "split": "train",
        "class_name": "pothole",
        "source": "sovit"
    })

for src in rdd_alligator_files:
    manifest_rows.append({
        "img_path": str(src),
        "split": "challenge_alligator",
        "class_name": "alligator",
        "source": "rdd"
    })

manifest = pd.DataFrame(manifest_rows)
manifest_path = output_root / "manifest.csv"
manifest.to_csv(manifest_path, index=False)

print("\nManifest saved to:", manifest_path)

# =========================================================
# 12. FINAL COUNT CHECK
# =========================================================
print("\nFinal dataset counts")
for split in ["train", "val", "test"]:
    pos_n = count_images(output_root / split / "pothole")
    neg_n = count_images(output_root / split / "non_pothole")
    print(f"{split}: pothole={pos_n}, non_pothole={neg_n}")

challenge_n = count_images(output_root / "challenge_alligator" / "alligator")
print("challenge_alligator:", challenge_n)

RDD pothole: 3674
RDD clean negative: 13224
RDD alligator challenge: 8412

RDD split sizes
train: 13518
val  : 1690
test : 1690

Sovit positive images found: 1119

RDD exported
train: {'pothole': 2939, 'non_pothole': 10579}
val  : {'pothole': 368, 'non_pothole': 1322}
test : {'pothole': 367, 'non_pothole': 1323}

Sovit positives added to train/pothole: 1119
Alligator challenge images: 8412

Manifest saved to: D:\Big_Data\2020YOLO\final_cnn_dataset\manifest.csv

Final dataset counts
train: pothole=4058, non_pothole=10579
val: pothole=368, non_pothole=1322
test: pothole=367, non_pothole=1323
challenge_alligator: 8412


In [3]:
from pathlib import Path

root = Path(r"D:\Big_Data\2020YOLO\final_cnn_dataset")

for split in ["train", "val", "test"]:
    pothole_n = len(list((root / split / "pothole").glob("*")))
    non_pothole_n = len(list((root / split / "non_pothole").glob("*")))
    print(split, "pothole =", pothole_n, "non_pothole =", non_pothole_n)

challenge_n = len(list((root / "challenge_alligator" / "alligator").glob("*")))
print("challenge_alligator =", challenge_n)

train pothole = 4058 non_pothole = 10579
val pothole = 368 non_pothole = 1322
test pothole = 367 non_pothole = 1323
challenge_alligator = 8412


In [4]:
import os
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score, precision_score

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

data_root = Path(r"D:\Big_Data\2020YOLO\final_cnn_dataset")

train_dir = data_root / "train"
val_dir   = data_root / "val"
test_dir  = data_root / "test"
challenge_dir = data_root / "challenge_alligator"

Using device: cuda


In [6]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(8),
    transforms.ColorJitter(brightness=0.12, contrast=0.12),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [7]:
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(val_dir, transform=eval_transform)
test_dataset  = datasets.ImageFolder(test_dir, transform=eval_transform)

print("train class_to_idx:", train_dataset.class_to_idx)

batch_size = 64   # 显存不够就改 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

train class_to_idx: {'non_pothole': 0, 'pothole': 1}


In [8]:
use_class_weights = False

num_pothole = 4058
num_non_pothole = 10579
total = num_pothole + num_non_pothole

weight_non_pothole = total / (2 * num_non_pothole)
weight_pothole = total / (2 * num_pothole)

class_to_idx = train_dataset.class_to_idx
weights = torch.zeros(2, dtype=torch.float32)
weights[class_to_idx["non_pothole"]] = weight_non_pothole
weights[class_to_idx["pothole"]] = weight_pothole
weights = weights.to(device)

print("Suggested class weights:", weights)

Suggested class weights: tensor([0.6918, 1.8035], device='cuda:0')


In [9]:
weights_enum = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights_enum)

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 2)

model = model.to(device)

In [10]:
if use_class_weights:
    criterion = nn.CrossEntropyLoss(weight=weights)
else:
    criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

In [11]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, loader, criterion, device, pothole_idx):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    all_labels = []
    all_preds = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    f1 = f1_score(all_labels, all_preds, pos_label=pothole_idx)
    recall = recall_score(all_labels, all_preds, pos_label=pothole_idx)
    precision = precision_score(all_labels, all_preds, pos_label=pothole_idx)

    return epoch_loss, epoch_acc, f1, recall, precision, all_labels, all_preds

In [12]:
pothole_idx = train_dataset.class_to_idx["pothole"]

num_epochs = 10
best_val_f1 = -1
save_path = "best_resnet18_rdd_sovit.pth"

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )

    val_loss, val_acc, val_f1, val_recall, val_precision, _, _ = evaluate(
        model, val_loader, criterion, device, pothole_idx
    )

    scheduler.step(val_f1)

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print(f"Val Precision (pothole): {val_precision:.4f}")
    print(f"Val Recall    (pothole): {val_recall:.4f}")
    print(f"Val F1        (pothole): {val_f1:.4f}")
    print("-" * 60)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), save_path)
        print(f"Best model saved to {save_path}")

Epoch [1/10]
Train Loss: 0.2842 | Train Acc: 0.8879
Val   Loss: 0.2246 | Val   Acc: 0.9154
Val Precision (pothole): 0.8947
Val Recall    (pothole): 0.6929
Val F1        (pothole): 0.7810
------------------------------------------------------------
Best model saved to best_resnet18_rdd_sovit.pth
Epoch [2/10]
Train Loss: 0.1927 | Train Acc: 0.9279
Val   Loss: 0.2217 | Val   Acc: 0.9231
Val Precision (pothole): 0.8673
Val Recall    (pothole): 0.7636
Val F1        (pothole): 0.8121
------------------------------------------------------------
Best model saved to best_resnet18_rdd_sovit.pth
Epoch [3/10]
Train Loss: 0.1564 | Train Acc: 0.9423
Val   Loss: 0.1954 | Val   Acc: 0.9296
Val Precision (pothole): 0.8761
Val Recall    (pothole): 0.7880
Val F1        (pothole): 0.8298
------------------------------------------------------------
Best model saved to best_resnet18_rdd_sovit.pth
Epoch [4/10]
Train Loss: 0.1336 | Train Acc: 0.9502
Val   Loss: 0.2343 | Val   Acc: 0.9225
Val Precision (pothol

In [13]:
model.load_state_dict(torch.load("best_resnet18_rdd_sovit.pth", map_location=device))

test_loss, test_acc, test_f1, test_recall, test_precision, y_true, y_pred = evaluate(
    model, test_loader, criterion, device, pothole_idx
)

print("Test Loss:", round(test_loss, 4))
print("Test Acc :", round(test_acc, 4))
print("Test Precision (pothole):", round(test_precision, 4))
print("Test Recall    (pothole):", round(test_recall, 4))
print("Test F1        (pothole):", round(test_f1, 4))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=test_dataset.classes))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Test Loss: 0.1923
Test Acc : 0.9373
Test Precision (pothole): 0.8595
Test Recall    (pothole): 0.8501
Test F1        (pothole): 0.8548

Classification Report:
              precision    recall  f1-score   support

 non_pothole       0.96      0.96      0.96      1323
     pothole       0.86      0.85      0.85       367

    accuracy                           0.94      1690
   macro avg       0.91      0.91      0.91      1690
weighted avg       0.94      0.94      0.94      1690

Confusion Matrix:
[[1272   51]
 [  55  312]]


In [14]:
challenge_dataset = datasets.ImageFolder(challenge_dir, transform=eval_transform)
challenge_loader = DataLoader(
    challenge_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print("challenge class_to_idx:", challenge_dataset.class_to_idx)

challenge class_to_idx: {'alligator': 0}


In [15]:
@torch.no_grad()
def evaluate_challenge_false_positive(model, loader, device, pothole_idx):
    model.eval()

    total = 0
    predicted_pothole = 0
    pothole_probs = []

    for images, _ in loader:
        images = images.to(device, non_blocking=True)

        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        preds = outputs.argmax(dim=1)

        predicted_pothole += (preds == pothole_idx).sum().item()
        pothole_probs.extend(probs[:, pothole_idx].cpu().numpy().tolist())
        total += images.size(0)

    fp_rate = predicted_pothole / total
    mean_prob = float(np.mean(pothole_probs))
    median_prob = float(np.median(pothole_probs))

    return fp_rate, mean_prob, median_prob, predicted_pothole, total

In [16]:
fp_rate, mean_prob, median_prob, predicted_pothole, total = evaluate_challenge_false_positive(
    model, challenge_loader, device, pothole_idx
)

print("Challenge alligator total:", total)
print("Predicted as pothole:", predicted_pothole)
print("False positive rate on alligator:", round(fp_rate, 4))
print("Mean pothole probability:", round(mean_prob, 4))
print("Median pothole probability:", round(median_prob, 4))

Challenge alligator total: 8412
Predicted as pothole: 4021
False positive rate on alligator: 0.478
Mean pothole probability: 0.4879
Median pothole probability: 0.4123
